# Homework 04 - Evaluation Answers

Notebook de respuestas para la Homework 4 del LLM Zoomcamp 2026.

Consigna verificada contra la cohorte 2026:

https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/04-evaluation/homework.md

Este notebook incluye el codigo Python para reproducir las respuestas principales: carga de datos, chunking, text search, vector search, hybrid search, Hit Rate y MRR.

## Final Answers

| Question | Answer |
|---|---|
| Q1 | `1400` |
| Q2 | `01-agentic-rag/lessons/03-rag.md` |
| Q3 | `01-agentic-rag/lessons/01-intro.md` |
| Q4 | `0.76` |
| Q5 | `0.55` |
| Q6 | `1` |


## 0. Dependencies

Si el entorno no tiene las librerias, instalar:

```bash
uv add gitsource minsearch numpy pandas tqdm onnxruntime tokenizers huggingface-hub openai pydantic python-dotenv
```

O desde notebook, usando el Python del kernel activo:

```python
import sys
!{sys.executable} -m pip install gitsource minsearch numpy pandas tqdm onnxruntime tokenizers huggingface-hub openai pydantic python-dotenv
```

## 1. Imports

In [13]:
import importlib.util
import subprocess
import sys

required_packages = {
    "gitsource": "gitsource",
    "minsearch": "minsearch",
    "numpy": "numpy",
    "pandas": "pandas",
    "tqdm": "tqdm",
    "onnxruntime": "onnxruntime",
    "tokenizers": "tokenizers",
    "huggingface_hub": "huggingface-hub",
    "openai": "openai",
    "pydantic": "pydantic",
    "dotenv": "python-dotenv",
}

missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]

print("Notebook Python:", sys.executable)

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are already available in this kernel.")

from pathlib import Path

import numpy as np
import pandas as pd
import onnxruntime as ort
from gitsource import GithubRepositoryDataReader, chunk_documents
from huggingface_hub import hf_hub_download
from minsearch import Index, VectorSearch
from tokenizers import Tokenizer
from tqdm.auto import tqdm

Notebook Python: c:\Users\TALIGENT\anaconda3\python.exe
All required packages are already available in this kernel.


## 2. Load Course Lesson Pages

La consigna fija el commit `8c1834d` para que todos trabajen con las mismas 72 paginas.

In [14]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

len(documents), [doc["filename"] for doc in documents[:3]]

(72,
 ['01-agentic-rag/lessons/01-intro.md',
  '01-agentic-rag/lessons/02-environment.md',
  '01-agentic-rag/lessons/03-rag.md'])

## 3. Q1 - Generate Questions and Read Input Tokens

Q1 requiere llamar a un LLM para generar preguntas de las primeras 3 paginas y leer `usage.input_tokens`.

El bloque siguiente es reproducible si tenes `OPENAI_API_KEY` configurada. Como consume API, queda apagado por defecto con `RUN_LLM_Q1 = False`.

In [16]:
RUN_LLM_Q1 = False

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.
Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

def build_data_generation_prompt(doc):
    return {
        "filename": doc["filename"],
        "content": doc["content"],
    }

if RUN_LLM_Q1:
    import os
    from dotenv import load_dotenv
    from openai import OpenAI
    from pydantic import BaseModel

    load_dotenv()
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    class Questions(BaseModel):
        questions: list[str]

    input_tokens = []
    generated_records = []

    for doc in documents[:3]:
        response = client.responses.parse(
            model="gpt-5.4-mini",
            input=[
                {"role": "system", "content": data_gen_instructions},
                {"role": "user", "content": str(build_data_generation_prompt(doc))},
            ],
            text_format=Questions,
        )

        parsed = response.output_parsed
        usage = response.usage
        input_tokens.append(usage.input_tokens)

        for question in parsed.questions:
            generated_records.append({
                "question": question,
                "filename": doc["filename"],
            })

    q1_average_input_tokens = sum(input_tokens) / len(input_tokens)
    print(input_tokens)
    print(q1_average_input_tokens)

# Closest option from the official choices.
q1_answer = "1400"
print("Q1 answer:", q1_answer)

Q1 answer: 1400


## 4. Load Official Ground Truth

Para Q2-Q6 no hace falta generar todo con LLM. La consigna provee `ground-truth.csv` con 360 preguntas.

In [17]:
ground_truth_url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv"
ground_truth = pd.read_csv(ground_truth_url).to_dict(orient="records")

ground_truth[0], len(ground_truth)

({'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 360)

## 5. Chunk Documents

Se usan los mismos chunks de HW2: ventanas de 2000 caracteres con step 1000.

In [18]:
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

## 6. Lightweight ONNX Embedder

Este bloque define el mismo embedder ONNX usado en HW2, pero autocontenido en el notebook. Descarga `Xenova/all-MiniLM-L6-v2` desde HuggingFace.

In [19]:
class Embedder:
    ONNX_CANDIDATES = [
        "onnx/model.onnx",
        "onnx/encoder_model.onnx",
        "model.onnx",
    ]

    def __init__(self, repo_id="Xenova/all-MiniLM-L6-v2", local_dir="models/Xenova/all-MiniLM-L6-v2"):
        import shutil
        from huggingface_hub import list_repo_files

        local_dir = Path(local_dir)
        local_dir.mkdir(parents=True, exist_ok=True)

        tokenizer_path = local_dir / "tokenizer.json"
        model_path = local_dir / "model.onnx"

        if not tokenizer_path.exists() or not model_path.exists():
            files = list_repo_files(repo_id=repo_id)
            onnx_file = next((candidate for candidate in self.ONNX_CANDIDATES if candidate in files), None)
            if onnx_file is None:
                raise FileNotFoundError(f"No ONNX model found in {repo_id}. Available files include: {files[:20]}")

            downloaded_tokenizer = hf_hub_download(repo_id=repo_id, filename="tokenizer.json")
            downloaded_model = hf_hub_download(repo_id=repo_id, filename=onnx_file)

            shutil.copy2(downloaded_tokenizer, tokenizer_path)
            shutil.copy2(downloaded_model, model_path)

            onnx_data_file = onnx_file + "_data"
            if onnx_data_file in files:
                downloaded_data = hf_hub_download(repo_id=repo_id, filename=onnx_data_file)
                shutil.copy2(downloaded_data, local_dir / "model.onnx_data")

        self.tokenizer = Tokenizer.from_file(str(tokenizer_path))
        self.session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        self.input_names = {inp.name for inp in self.session.get_inputs()}

    def encode(self, text, normalize=True):
        return self.encode_batch([text], normalize=normalize)[0]

    def encode_batch(self, texts, normalize=True):
        self.tokenizer.enable_padding()
        encoded = self.tokenizer.encode_batch(texts)

        feed = {}
        if "input_ids" in self.input_names:
            feed["input_ids"] = np.array([e.ids for e in encoded], dtype=np.int64)
        if "attention_mask" in self.input_names:
            feed["attention_mask"] = np.array([e.attention_mask for e in encoded], dtype=np.int64)
        if "token_type_ids" in self.input_names:
            feed["token_type_ids"] = np.array([e.type_ids for e in encoded], dtype=np.int64)

        hidden = self.session.run(None, feed)[0]
        mask = feed["attention_mask"][..., None]
        pooled = (hidden * mask).sum(axis=1) / mask.sum(axis=1)

        if normalize:
            pooled = pooled / np.linalg.norm(pooled, axis=1, keepdims=True)

        return pooled

## 7. Build Text and Vector Search

In [20]:
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

embedder = Embedder()
chunk_vectors = embedder.encode_batch([chunk["content"] for chunk in tqdm(chunks)])
X = np.vstack(chunk_vectors)

vector_index = VectorSearch(keyword_fields={"filename"})
vector_index.fit(X, chunks)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vector_index.search(query_vector, num_results=num_results)

def filenames(results):
    return [result["filename"] for result in results]

  0%|          | 0/295 [00:00<?, ?it/s]

## 8. Q2 and Q3 - First Result for the First Ground Truth Question

In [21]:
q = ground_truth[0]["question"]

text_results = text_search(q)
vector_results = vector_search(q)

q2_answer = text_results[0]["filename"]
q3_answer = vector_results[0]["filename"]

print("Question:", q)
print("Q2 text top:", q2_answer)
print("Q3 vector top:", q3_answer)

Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
Q2 text top: 01-agentic-rag/lessons/03-rag.md
Q3 vector top: 01-agentic-rag/lessons/01-intro.md


## 9. Evaluation Functions

Un resultado cuenta como relevante si el `filename` del chunk devuelto coincide con el `filename` esperado en ground truth.

In [22]:
def compute_relevance(record, search_function):
    results = search_function(record["question"])
    return [int(result["filename"] == record["filename"]) for result in results]

def hit_rate(relevance_total):
    return sum(any(line) for line in relevance_total) / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank, rel in enumerate(line):
            if rel == 1:
                total_score += 1 / (rank + 1)
                break

    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []

    for record in tqdm(ground_truth):
        relevance = compute_relevance(record, search_function)
        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## 10. Q4 and Q5 - Evaluate Text and Vector Search

In [23]:
text_metrics = evaluate(ground_truth, text_search)
vector_metrics = evaluate(ground_truth, vector_search)

q4_answer = "0.76"  # closest to text_metrics['hit_rate']
q5_answer = "0.55"  # closest to vector_metrics['mrr']

print("Text metrics:", text_metrics)
print("Vector metrics:", vector_metrics)
print("Q4 answer:", q4_answer)
print("Q5 answer:", q5_answer)

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

Text metrics: {'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}
Vector metrics: {'hit_rate': 0.725, 'mrr': 0.5486111111111112}
Q4 answer: 0.76
Q5 answer: 0.55


## 11. Q6 - Hybrid Search with RRF

In [24]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def make_hybrid_search(k):
    def hybrid_search(query, num_results=5):
        text_results = text_search(query, num_results=10)
        vector_results = vector_search(query, num_results=10)
        return rrf([text_results, vector_results], k=k, num_results=num_results)

    return hybrid_search

In [25]:
hybrid_metrics = {}

for k in [1, 50, 100, 200]:
    metrics = evaluate(ground_truth, make_hybrid_search(k))
    hybrid_metrics[k] = metrics
    print(k, metrics)

# Pick the k with best MRR. If there is a tie, pick the smallest k.
q6_answer = min(
    hybrid_metrics,
    key=lambda k: (-hybrid_metrics[k]["mrr"], k),
)

print("Q6 answer:", q6_answer)

  0%|          | 0/360 [00:00<?, ?it/s]

1 {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


  0%|          | 0/360 [00:00<?, ?it/s]

50 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

100 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

200 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
Q6 answer: 1


## 12. Submission Dictionary

In [26]:
answers = {
    "Q1": q1_answer,
    "Q2": q2_answer,
    "Q3": q3_answer,
    "Q4": q4_answer,
    "Q5": q5_answer,
    "Q6": str(q6_answer),
}

answers

{'Q1': '1400',
 'Q2': '01-agentic-rag/lessons/03-rag.md',
 'Q3': '01-agentic-rag/lessons/01-intro.md',
 'Q4': '0.76',
 'Q5': '0.55',
 'Q6': '1'}